# Lab 2b · Tu CV, detrás de un servidor MCP

**MCP (Model Context Protocol)** es un estándar abierto para conectar agentes con datos y sistemas: escribes un
**servidor** una vez y lo usa cualquier **cliente** (tu agente, GitHub Copilot, Claude…). El nuestro,
[`servidor_mcp_carrera.py`](servidor_mcp_carrera.py), pone tu carrera al alcance del agente… con reglas:

| Herramienta | Qué hace | ¿Escribe? |
|---|---|---|
| `leer_cv` | Tu CV en Markdown | No |
| `plan_de_estudios` | Las asignaturas oficiales del MUCD-IA, con su estado hoy (cursada, en curso, pendiente) | No |
| `buscar_competencias` | Dónde aprendiste algo: tu grado, tu máster, el plan y las masterclasses | No |
| `guardar_cv` | Guarda una versión nueva (Markdown y Word)… **si no inventa nada** | Sí, solo en `sesion2/trabajo/` |

Tiempo: 20 minutos. El código ya está resuelto: lee cada pieza ✏️ y ejecuta.

In [ ]:
import sys

import pandas as pd

import mucdia

sys.path.insert(0, str(mucdia.config.RAIZ / "sesion2"))
import candidatura as C  # noqa: E402

PIEZAS = C.piezas.CARPETA      # tus celdas ✏️ guardan aquí su código: lo usará tu webapp
PIEZAS.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 110)
cliente = mucdia.cliente()

oferta = C.oferta.cargar_oferta()          # la del lab 2a
print(f"Oferta: {oferta.puesto} · {oferta.empresa}")

## Paso 1 · Tu perfil (3')

El agente solo puede usar lo que conste en tu perfil, en `sesion2/mi_perfil/`:

- `cv.md` · tu CV. Arrastra tu PDF o Word a esa carpeta y conviértelo con la primera línea de la celda siguiente, o
  pega el texto en `cv.md`. (También puedes subirlo desde la webapp.)
- `grado.md` · lo que aprendiste en tu grado, y `master.md` · lo que llevas hecho en el máster (prácticas,
  proyectos). Copia las plantillas de [`ejemplo/`](ejemplo/) y escribe solo lo que puedas demostrar.

Si aún no has puesto tu CV, se usa el de **Lucía**, ficticia ([`ejemplo/cv.md`](ejemplo/cv.md)). Nunca se mezclan.

In [ ]:
# C.carrera.importar_cv(mucdia.config.RAIZ / "sesion2/mi_perfil/mi_cv.pdf")   # ✏️ opcional: tu CV en PDF o Word
yo = C.carrera.perfil()
print(f"Perfil: {yo['origen'].upper()} · CV: {C.fmt(len(yo['cv']))} caracteres · grado: {'sí' if yo['grado'] else 'vacío'} · "
      f"máster: {'sí' if yo['master'] else 'vacío'}\n")
print(yo["cv"][:700], "…")

## Paso 2 · MCP por dentro (sin modelo, sin cuota)

MCP es **JSON-RPC 2.0**: mensajes de texto, uno por línea, por la entrada y la salida estándar del proceso del
servidor (transporte *stdio*). `mucdia.ClienteMCPCrudo()` lanza el servidor y te enseña cada mensaje: **→** lo que
enviamos y **←** lo que responde. Es exactamente lo que hacen Copilot o Claude antes de dejar que el modelo use las
herramientas.

In [ ]:
mcp = mucdia.ClienteMCPCrudo(ancho=400)
mcp.peticion("initialize", {"protocolVersion": "2025-06-18", "capabilities": {},
                            "clientInfo": {"name": "lab2b", "version": "1.0"}})
mcp.notificacion("notifications/initialized")
herramientas = mcp.peticion("tools/list")["result"]["tools"]
mcp.peticion("tools/call", {"name": "buscar_competencias", "arguments": {"consulta": "Python"}})
mcp.cerrar()
for h in herramientas:
    print(f"{'📖' if h['annotations'].get('readOnlyHint') else '✍️ '} {h['name']}: {h['description'][:90]}…")

🤔 Solo una herramienta escribe, y solo en su carpeta: **mínimo privilegio**. Un servidor MCP es software que se
ejecuta con tus permisos; si pudiera borrar, tu agente podría borrar.

## Paso 3 · ✏️ El guardarraíl: un control en código, dentro del servidor (5')

Pídele a un modelo que «mejore» tu CV y tenderá a inflarlo: de 5 horas a 6, de Excel a Tableau. A un modelo se le
puede convencer; **a una función de Python, no**. `cifras_sin_respaldo(cv_nuevo, fuentes)` devuelve las cifras del CV
nuevo que no aparecen en ninguna fuente (tu CV, tu grado, tu máster). «1.200» y «1200» cuentan como la misma.

`guardar_cv` la ejecuta antes de guardar (junto con un control de herramientas como Tableau o Azure): si encuentra
algo, **no guarda** y devuelve un ERROR que el agente tiene que corregir.

In [ ]:
%%writefile "{PIEZAS}/guardarrail.py"
import re


def cifras_sin_respaldo(cv_nuevo: str, fuentes: str) -> list[str]:
    """Cifras del CV nuevo que no aparecen en ninguna fuente («1.200» y «1200» cuentan como la misma)."""
    def normalizar(cifra):
        return re.sub(r"[.,]", "", cifra)

    conocidas = {normalizar(c) for c in re.findall(r"\d+(?:[.,]\d+)*", fuentes)}
    nuevas = []
    for cifra in re.findall(r"\d+(?:[.,]\d+)*", cv_nuevo):
        if normalizar(cifra) not in conocidas and cifra not in nuevas:
            nuevas.append(cifra)
    return nuevas

In [ ]:
cifras_sin_respaldo = C.piezas.cargar("guardarrail", "cifras_sin_respaldo")
if mucdia.lab.comprobar_respaldo(cifras_sin_respaldo):
    # Y ahora, de verdad: pedimos al servidor (otro proceso) que guarde un CV inflado
    inflado = yo["cv"] + "\n- Lideré un equipo de 12 personas y aumenté las ventas un 35 % con Tableau."
    with mucdia.ClienteMCPCrudo(ancho=320) as mcp:
        mcp.peticion("initialize", {"protocolVersion": "2025-06-18", "capabilities": {},
                                    "clientInfo": {"name": "lab2b", "version": "1.0"}})
        mcp.notificacion("notifications/initialized")
        mcp.peticion("tools/call", {"name": "guardar_cv", "arguments": {"empresa": "prueba", "puesto": "prueba",
                                                                        "cv_markdown": inflado}})

## Paso 4 · ✏️ El puesto de trabajo del editor (4')

El guardarraíl atrapa cifras y herramientas; el resto depende de las **instrucciones**. Estas son las del editor de
CV: cómo trabajar (con qué herramientas y en qué orden), qué puede cambiar, qué no puede hacer nunca y qué hacer si
`guardar_cv` devuelve ERROR. Se guardan en `sesion2/mis_piezas/instrucciones.json` y las usará tu webapp.

In [ ]:
INSTRUCCIONES_EDITOR = """Eres el editor de CV de una persona que busca empleo en datos e IA. Adaptas su CV a UNA oferta
sin inventar nada: un CV con una mentira es peor que un CV peor escrito.

Cómo trabajas:
1. Lee el CV con leer_cv y el estado del máster con plan_de_estudios.
2. Para cada requisito técnico de la oferta, busca pruebas con buscar_competencias.
3. Antes de escribir, repasa las palabras clave de la oferta una a una: «la uso» si describe con verdad algo que la
   persona ha hecho o está estudiando (análisis de datos para quien ha analizado datos, por ejemplo), o «hueco» si no.
   Un filtro automático (ATS) busca esas palabras EXACTAS: escribe tal cual las que uses.
4. Reescribe el CV COMPLETO en Markdown, con su formato (nombre como título, contacto y secciones) y en el idioma de la
   oferta. Adáptalo de verdad, no lo copies:
   - Un único perfil de 2 o 3 líneas orientado al puesto, con las palabras clave que uses.
   - Ordena secciones, logros y proyectos para que lo más relevante para la oferta salga primero. No añadas logros
     nuevos: reescribe y reordena los que hay.
   - Sin inflar: 1.200 reseñas no son «grandes conjuntos de datos», y unas prácticas no son «un equipo
     multidisciplinar y ágil» si el CV no lo dice.
   - Bajo el máster, añade las asignaturas EN CURSO que más encajen con la oferta (como formación en curso) y, si
     encajan, las masterclasses ya HECHAS. Nunca las pendientes.
   - Prohibido inventar o inflar experiencia, cifras, títulos, herramientas o niveles que no consten en el CV, el grado
     o el máster.
5. Guarda el CV con guardar_cv. Si devuelve ERROR, quita o reescribe lo que señala y vuelve a guardarlo (como mucho,
   3 intentos).
6. Responde con el archivo que devolvió guardar_cv, las palabras clave usadas, los cambios, el encaje de cada requisito
   imprescindible y los huecos. Sé exigente con el encaje: «cubierto» solo si la prueba cumple el requisito tal como
   lo pide la oferta; si se parece o es discutible, «parcial». Lo que pongas en huecos no puede estar «cubierto». Un
   diagnóstico honesto sirve más que uno amable."""
C.piezas.guardar_instrucciones("editor_cv", INSTRUCCIONES_EDITOR)
mucdia.lab.comprobar_editor(INSTRUCCIONES_EDITOR)

## Paso 5 · El agente adapta tu CV por MCP (3')

Lanzamos el servidor (`mucdia.servidor_mcp()`) y se lo damos al agente como herramienta. Mira en la traza cómo
descubre las herramientas, busca pruebas… y qué pasa si `guardar_cv` le devuelve un ERROR.

In [ ]:
servidor = mucdia.servidor_mcp()                    # lanza sesion2/servidor_mcp_carrera.py por stdio
editor = C.agentes.editor_cv(cliente, servidor)     # con tus instrucciones (mis_piezas/instrucciones.json)
mucdia.reiniciar_traza()
r = await editor.run("Adapta mi CV a esta oferta:\n\n" + C.agentes.describir_oferta(oferta),
                     options={"response_format": C.agentes.CVAdaptado})
await servidor.close()
adaptado = r.value
mucdia.mostrar_traza(r)

In [ ]:
from IPython.display import Markdown, display

ruta = C.carrera.ultimo_cv()
cv_nuevo = ruta.read_text(encoding="utf-8")
print(f"📄 {ruta.relative_to(mucdia.config.RAIZ)} (y su versión en Word, .docx)\n")
display(Markdown(cv_nuevo))

## Paso 6 · Mide: ¿mejor y sin mentir?

Dos números: la **cobertura** de palabras clave de la oferta (lo que miraría un filtro automático), antes y
después, y las **afirmaciones sin respaldo** del CV guardado (deben ser 0).

In [ ]:
antes = C.carrera.cobertura(oferta.palabras_clave, yo["cv"])
despues = C.carrera.cobertura(oferta.palabras_clave, cv_nuevo)
sin_respaldo = C.carrera.afirmaciones_sin_respaldo(cv_nuevo)
print(f"📏 Cobertura de palabras clave: {antes['porcentaje']} % → {despues['porcentaje']} % · "
      f"afirmaciones sin respaldo: {len(sin_respaldo)} · huecos: {len(adaptado.huecos)}")
print("   Siguen faltando:", ", ".join(despues["ausentes"]) or "nada")
pd.DataFrame([e.model_dump() for e in adaptado.encaje])

🤔 Para pensar:
- **Léelo con lupa.** ¿Exagera algo que el guardarraíl no puede ver (convertir 1.200 reseñas en «grandes conjuntos
  de datos», unas prácticas en «equipo multidisciplinar»)? El CV lo firmas tú, no el agente.
- Una cobertura del 100 % ¿sería buena señal? ¿Qué le pasaría al CV de quien la persiga a toda costa?
- Los huecos no van al CV: van a tu plan (lab 2c).

## ⭐ Para ir más allá

1. **El mismo servidor en GitHub Copilot.** [.vscode/mcp.json](../.vscode/mcp.json) ya registra `mi-carrera`. En
   Copilot Chat, modo **Agent**, actívalo y pídele: *«Adapta mi CV a esta oferta: …»*. Mismo servidor, mismo
   guardarraíl, otro cliente.
2. **El inspector oficial de MCP**: `mcp dev sesion2/servidor_mcp_carrera.py` (necesita Node.js).
3. Añade al servidor una herramienta `historial()` que liste las versiones guardadas. ¿Es de solo lectura?